In [29]:
##### IMPORTS

import os
os.environ['picaso_refdata'] = r'C:\Users\Alex\Desktop\Picaso\picaso\reference' # THIS MUST GO BEFORE YOUR IMPORT STATEMENT
os.environ['PYSYN_CDBS'] = r'C:\Users\Alex\Desktop\Picaso\grp\redcat\trds' # This is for the stellar data discussed below.

# General
import numpy as np
import astropy.units as u
import matplotlib.pyplot as plt
import bd_support as sup

from pathlib import Path
from itertools import product

# Picaso and Virga
from picaso import justdoit as jdi
from virga import justdoit as vj

from bokeh.io import output_notebook
output_notebook()

# To see what clouds are availible
# vj.available()

Loading BokehJS ...

In [30]:
##### CONFIGURATIONS

# Directories
sonor_path  = r'C:\Users\Alex\Desktop\Picaso\data\sonora' # Sonora db
# sonor_path  = '/groups/tkaralidi/pbraunschweig/training_set/profiles/'
virga_path  = r'C:\Users\Alex\Desktop\Picaso\data\virga'  # Virga
# virga_path  = '/home/sa221179/picaso/virga/'
opaci_path  = None # Opacity db
# opcai_path  = '/groups/tkaralidi/opacity_500k_for_R5000_egpoutput.db'

# Constant values
wav_range   = [0.3, 5.0] # microns
MH          = 1.0        # [M/H] metallicity factor ~ solar
MU          = 2.36       # Average MU
R           = 300        # resolution
# R           = 5000

In [31]:
##### MANUAL TESTING

# I vary Teff for every 50
# Keeping logg and fsed the same
Teff = 1400
logg = 4.0
fsed = 2.0
kzz = 1e9

# Run
opa    = jdi.opannection(wav_range, opaci_path)
bd     = jdi.inputs(calculation="browndwarf")
bd.phase_angle(0)
gravity = 10**logg * 1e-2
bd.gravity(gravity, gravity_unit=u.Unit('m/s**2'))
bd.sonora(sonor_path, Teff)
sup.inject_corr(bd, Teff, logg, fsed)

prof = bd.inputs['atmosphere']['profile']
P = np.asarray(prof['pressure'], float)
T = np.asarray(prof['temperature'], float)
bd.inputs["atmosphere"]["profile"]["kz"] = [float(kzz)] * len(P)

# View Rec
rec = vj.recommend_gas(P, T, MH, MU, plot=True, legend='inside')
print(rec)

['Al2O3', 'CaAl12O19', 'CaTiO3', 'Cr', 'Fe', 'Mg2SiO4', 'MgSiO3', 'MnS', 'Na2S', 'SiO2', 'TiO2']


In [32]:
cloud_dict   = {'KCl' : '1', 'Na2S' : '2', 'MnS' : '3', 'ZnS' : '4', 'Cr' : '5',
                'Fe' : '6', 'Mg2SiO4' : '7', 'MgSiO3' : '8', 'Al2O3' : '9'}
# Read log for explanation
hard_exclude = {'CaAl12O19', 'SiO2', 'H2O', 'NH3', 'TiO2', 'CH4', 'CaTiO3'}

Tdwarf = {'KCl', 'Na2S', 'MnS', 'ZnS', 'Cr'}
Ldwarf = {'Fe', 'Mg2SiO4', 'MgSiO3', 'Al2O3'}
usecl  = Ldwarf if Teff > 1300 else Tdwarf
clouds = [sp for sp in rec if sp in usecl and sp not in hard_exclude]

In [33]:
print(clouds)

['Al2O3', 'Fe', 'Mg2SiO4', 'MgSiO3']
